In [ ]:
import sys, os
import pickle, ssl
from flask import Flask, request

_HERE         = os.path.abspath(os.getcwd())
_PROJECT_ROOT = os.path.dirname(_HERE)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from config import (
    SERVER_PORT, CERT_FILE, KEY_FILE, CHUNKS_FOLDER
)

os.makedirs(CHUNKS_FOLDER, exist_ok=True)

app = Flask(__name__)

@app.route("/", methods=["POST"])
def receive_chunk():
    try:
        payload   = pickle.loads(request.data)
        chunk_id  = payload["chunk_id"]
        client_id = payload["client_id"]
        data      = payload["data"]

        if not all([chunk_id is not None, client_id, data is not None]):
            print(f"[SERVER] Invalid payload structure")
            return "Bad Request", 400

        client_folder = os.path.join(CHUNKS_FOLDER, client_id)
        os.makedirs(client_folder, exist_ok=True)

        filename = os.path.join(client_folder, f"chunk_{chunk_id}.bin")
        with open(filename, "wb") as f:
            pickle.dump({"data": data}, f)

        print(f"[SERVER] ✓ Received raw chunk {chunk_id} from {client_id}")
        return "OK", 200

    except Exception as e:
        print(f"[SERVER] Error: {e}")
        return "Error", 500

@app.route("/health", methods=["GET"])
def health():
    return "OK", 200

def run_server():
    if not os.path.exists(CERT_FILE):
        print(f"[SERVER] ERROR: cert.pem not found at {CERT_FILE}")
        return

    ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    ssl_context.load_cert_chain(certfile=CERT_FILE, keyfile=KEY_FILE)

    print(f"[SERVER] Chunks folder : {CHUNKS_FOLDER}")
    print(f"[SERVER] Listening securely on https://127.0.0.1:{SERVER_PORT} ...")
    app.run(
        host="127.0.0.1",
        port=SERVER_PORT,
        ssl_context=ssl_context,
        threaded=True,
    )

run_server()


[SERVER] Chunks folder : /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/server/received_chunks_bin
[SERVER] Listening securely on https://127.0.0.1:5055 ...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on https://127.0.0.1:5055
Press CTRL+C to quit
127.0.0.1 - - [30/Jun/2026 17:34:54] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:34:54] "POST / HTTP/1.1" 200 -


[SERVER] ✓ Received raw chunk 0 from hospital_2
[SERVER] ✓ Received raw chunk 1 from hospital_2


127.0.0.1 - - [30/Jun/2026 17:34:55] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:34:55] "POST / HTTP/1.1" 200 -


[SERVER] ✓ Received raw chunk 0 from hospital_1
[SERVER] ✓ Received raw chunk 1 from hospital_1


127.0.0.1 - - [30/Jun/2026 17:35:00] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/Jun/2026 17:35:00] "POST / HTTP/1.1" 200 -


[SERVER] ✓ Received raw chunk 0 from hospital_3
[SERVER] ✓ Received raw chunk 1 from hospital_3
